# System prompts don't guarantee tool use

System prompts say "you MUST use this tool." Does the model actually follow that?

Same ReAct agent, same task, five mandatory tools. Each experiment swaps the tool and tracks whether the model calls it. 10 runs per tool, two models (Flash Lite and Flash).

Tools tested:
1. Think — no-op scratchpad, returns nothing useful
2. Classify — categorizes the query, returns a label
3. Plan — writes a research plan, returns sub-questions
4. Validate — checks the answer, returns pass/fail
5. Log — audit logging, returns a confirmation ID

In [1]:
%%capture --no-stderr
%pip install --quiet -U langchain-core langgraph langgraph-prebuilt langchain-google-genai langchain-tavily pydantic

In [1]:
import os, getpass, json, statistics
from collections import Counter
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f'{var}: ')

_set_env('GOOGLE_API_KEY')
_set_env('TAVILY_API_KEY')

In [2]:
# Models
llm_lite = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite-preview')
llm_pro = ChatGoogleGenerativeAI(model='gemini-3-flash-preview')

# Base search tool
tavily_search = TavilySearch(max_results=5, topic='general')

N_RUNS = 10
TASK = "What are the key differences between LangGraph and CrewAI for building multi-agent systems in 2026?"

print(f"Models ready. Runs per experiment: {N_RUNS}")

Models ready. Runs per experiment: 10


---
## Tools

Five tools with different return values. The question is whether the model treats them all the same when told "you MUST."

In [3]:
# 1. THINK — pure no-op. Returns nothing the model needs.
@tool
def think(thought: str) -> str:
    """Use this to think through your research strategy, analyze what you've found so far,
    identify gaps, and plan your next search. This doesn't search anything — it's a place to reason."""
    return "Thought noted. Continue with your research."


# 2. CLASSIFY — categorizes the query. Returns a label the model could infer on its own.
@tool
def classify_query(query: str) -> str:
    """Classify the research query into a category before searching.
    Returns the category to help guide search strategy."""
    query_lower = query.lower()
    if any(w in query_lower for w in ['compare', 'vs', 'difference', 'better']):
        return "Category: COMPARISON — search for head-to-head analyses and benchmarks."
    elif any(w in query_lower for w in ['how to', 'build', 'implement', 'create']):
        return "Category: TUTORIAL — search for guides and documentation."
    elif any(w in query_lower for w in ['what is', 'define', 'explain']):
        return "Category: DEFINITION — search for authoritative definitions."
    else:
        return "Category: GENERAL — use broad search terms."


# 3. PLAN — writes a structured research plan. Returns actionable structure.
@tool
def research_plan(question: str) -> str:
    """Create a structured research plan before starting your search.
    Break the question into sub-questions that need to be answered."""
    # Simulate a plan (in reality, a smarter version could use an LLM)
    return (
        f"Research plan for: {question}\n"
        f"Sub-questions to investigate:\n"
        f"1. What are the core architectural differences?\n"
        f"2. What are the production readiness indicators?\n"
        f"3. What do practitioners report about each?\n"
        f"Search for each sub-question separately, then synthesize."
    )


# 4. VALIDATE — checks the final answer for unsupported claims. Critical for correctness.
@tool
def validate_answer(answer: str, sources: str) -> str:
    """Validate your final answer before responding. Check that every claim
    is supported by a source. Flag any claims that aren't grounded in search results."""
    answer_len = len(answer.split())
    source_count = sources.count('http')
    issues = []
    if source_count == 0:
        issues.append("NO SOURCES — answer has zero citations.")
    if answer_len > 200 and source_count < 2:
        issues.append(f"UNDER-CITED — {answer_len} words but only {source_count} source(s).")
    if any(phrase in answer.lower() for phrase in ['everyone knows', 'obviously', 'clearly']):
        issues.append("HEDGE WORDS — contains unsupported confidence markers.")
    if not issues:
        return "VALIDATION PASSED — answer appears grounded in sources."
    return f"VALIDATION FAILED — {len(issues)} issue(s): " + " | ".join(issues)


# 5. LOG — audit logging. Returns confirmation. Policy requirement, not task requirement.
@tool
def log_query(query: str, timestamp: str) -> str:
    """Log this research query for audit and compliance purposes.
    Must be called before any search is performed. Returns a log confirmation ID."""
    import hashlib, time
    log_id = hashlib.md5(f"{query}{time.time()}".encode()).hexdigest()[:12]
    return f"LOGGED — ID: {log_id}. Proceed with research."


print("Tools defined:")
for t in [think, classify_query, research_plan, validate_answer, log_query]:
    print(f"  {t.name}: {t.description[:80]}...")

Tools defined:
  think: Use this to think through your research strategy, analyze what you've found so f...
  classify_query: Classify the research query into a category before searching.
Returns the catego...
  research_plan: Create a structured research plan before starting your search.
Break the questio...
  validate_answer: Validate your final answer before responding. Check that every claim
is supporte...
  log_query: Log this research query for audit and compliance purposes.
Must be called before...


---
## System prompts

Each prompt uses the same structure: "You MUST use [tool] before [searching / responding]." No ambiguity.

In [4]:
prompts = {
    'think': SystemMessage(content=(
        'You are a research assistant. Search for accurate, up-to-date information. '
        'IMPORTANT: Before EVERY search, you MUST use the think tool to reason about '
        'what you know so far and what you still need to find. Never search without '
        'thinking first. After gathering results, use think again to evaluate whether '
        'the sources agree or conflict. Cite your sources with URLs.'
    )),

    'classify': SystemMessage(content=(
        'You are a research assistant. Search for accurate, up-to-date information. '
        'IMPORTANT: You MUST use the classify_query tool on the question before doing '
        'anything else. This determines your search strategy. Do NOT search until you '
        'have classified the query. Cite your sources with URLs.'
    )),

    'plan': SystemMessage(content=(
        'You are a research assistant. Search for accurate, up-to-date information. '
        'IMPORTANT: You MUST use the research_plan tool to create a structured plan '
        'before performing any searches. Follow the plan step by step. Do NOT search '
        'until you have a plan. Cite your sources with URLs.'
    )),

    'validate': SystemMessage(content=(
        'You are a research assistant. Search for accurate, up-to-date information. '
        'IMPORTANT: Before giving your final answer, you MUST use the validate_answer tool '
        'to check that every claim is supported by a source. Pass your draft answer and '
        'sources to the validator. If validation fails, fix the issues and validate again. '
        'Never respond without validating first. Cite your sources with URLs.'
    )),

    'log': SystemMessage(content=(
        'You are a research assistant. Search for accurate, up-to-date information. '
        'IMPORTANT: For compliance, you MUST call log_query with the user\'s question '
        'and current timestamp before performing any search. This is a mandatory audit '
        'requirement. Do NOT search until the query is logged. Cite your sources with URLs.'
    )),
}

print("Prompts defined for:", list(prompts.keys()))

Prompts defined for: ['think', 'classify', 'plan', 'validate', 'log']


---
## Infrastructure

Graph builder and experiment runner. Tracks tool calls per run, flags skips.

In [5]:
run_stats = []

def build_graph(tools_list, sys_msg, model=None):
    _llm = model or llm_lite
    llm_with_tools = _llm.bind_tools(tools_list)

    def llm_node(state: MessagesState):
        response = llm_with_tools.invoke([sys_msg] + state['messages'])
        run_stats.append({
            'loop': len(run_stats) + 1,
            'input_tokens': response.usage_metadata['input_tokens'],
            'output_tokens': response.usage_metadata['output_tokens'],
            'tool_calls': [c['name'] for c in response.tool_calls],
            'calls_detail': [{'name': c['name'], 'args': {k: str(v)[:60] for k, v in c['args'].items()}} for c in response.tool_calls],
        })
        return {'messages': [response]}

    builder = StateGraph(MessagesState)
    builder.add_node('llm', llm_node)
    builder.add_node('tools', ToolNode(tools_list))
    builder.add_edge(START, 'llm')
    builder.add_conditional_edges('llm', tools_condition)
    builder.add_edge('tools', 'llm')
    return builder.compile()


def run_experiment(graph, task, mandatory_tool_name, n=N_RUNS, recursion_limit=25):
    """Run n times. Track whether the mandatory tool was used in each run."""
    all_runs = []
    for i in range(n):
        run_stats.clear()
        try:
            result = graph.invoke(
                {'messages': [HumanMessage(content=task)]},
                config={'recursion_limit': recursion_limit},
                version='v2'
            )
            stats = list(run_stats)

            # Flatten all tool calls across all loops
            all_tool_calls = []
            for s in stats:
                all_tool_calls.extend(s['tool_calls'])

            tool_counts = Counter(all_tool_calls)
            mandatory_used = tool_counts.get(mandatory_tool_name, 0)

            # Determine call order (was mandatory tool called when expected?)
            first_calls = stats[0]['tool_calls'] if stats else []
            mandatory_called_first = mandatory_tool_name in first_calls

            run_data = {
                'run': i + 1,
                'loops': len(stats),
                'total_tokens': sum(s['input_tokens'] + s['output_tokens'] for s in stats),
                'input_tokens': sum(s['input_tokens'] for s in stats),
                'output_tokens': sum(s['output_tokens'] for s in stats),
                'tool_counts': dict(tool_counts),
                'mandatory_count': mandatory_used,
                'mandatory_skipped': mandatory_used == 0,
                'mandatory_called_first': mandatory_called_first,
                'call_sequence': [c for s in stats for c in s['tool_calls']],
                'stats': stats,
                'crashed': False,
            }
            all_runs.append(run_data)

            skip_flag = ' ← SKIPPED' if mandatory_used == 0 else ''
            print(f"  Run {i+1}/{n}: {len(stats)} loops, {mandatory_tool_name}={mandatory_used}, tools={dict(tool_counts)}{skip_flag}")

        except Exception as e:
            all_runs.append({
                'run': i + 1, 'loops': len(run_stats), 'total_tokens': 0,
                'input_tokens': 0, 'output_tokens': 0,
                'crashed': True, 'error': str(e),
                'mandatory_count': 0, 'mandatory_skipped': True,
                'tool_counts': {}, 'call_sequence': [],
            })
            print(f"  Run {i+1}/{n}: CRASHED — {e}")

    return all_runs


def summarize_adherence(runs, tool_name, label):
    """Print adherence summary for one experiment."""
    ok = [r for r in runs if not r.get('crashed')]
    if not ok:
        print(f"\n{label}: ALL CRASHED")
        return

    skipped = sum(1 for r in ok if r['mandatory_skipped'])
    used = len(ok) - skipped
    called_first = sum(1 for r in ok if r.get('mandatory_called_first'))
    avg_count = statistics.mean([r['mandatory_count'] for r in ok])
    avg_tokens = statistics.mean([r['total_tokens'] for r in ok])
    avg_loops = statistics.mean([r['loops'] for r in ok])

    print(f"\n{'='*70}")
    print(f"{label}")
    print(f"{'='*70}")
    print(f"  Adherence:        {used}/{len(ok)} runs used {tool_name} ({used/len(ok)*100:.0f}%)")
    print(f"  Skipped:          {skipped}/{len(ok)} runs ({skipped/len(ok)*100:.0f}%)")
    print(f"  Called first:     {called_first}/{len(ok)} runs ({called_first/len(ok)*100:.0f}%)")
    print(f"  Avg times called: {avg_count:.1f}")
    print(f"  Avg loops:        {avg_loops:.1f}")
    print(f"  Avg tokens:       {avg_tokens:.0f}")

    # Show call sequences for first 3 runs
    print(f"  Sample sequences:")
    for r in ok[:3]:
        seq = ' → '.join(r['call_sequence']) if r['call_sequence'] else '(no tools)'
        print(f"    Run {r['run']}: {seq}")


print("Infrastructure ready.")

Infrastructure ready.


---
## Experiments

Same task, same agent loop. Only the mandatory tool changes.

| Experiment | Tool | What it returns |
|---|---|---|
| 1 | Think | "Thought noted" (nothing useful) |
| 2 | Classify | Category label |
| 3 | Plan | Sub-questions |
| 4 | Validate | Pass/fail with issues |
| 5 | Log | Confirmation ID |

---
### Experiment 1 — Think (no-op)

The model gains nothing from calling this. System prompt says: "Before EVERY search, you MUST use the think tool."

In [18]:
graph_think = build_graph([tavily_search, think], prompts['think'])
print(f"Experiment 1 — Think Tool (Flash Lite, {N_RUNS} runs)")
runs_think_lite = run_experiment(graph_think, TASK, 'think')
summarize_adherence(runs_think_lite, 'think', 'Think Tool — Flash Lite')

Experiment 1 — Think Tool (Flash Lite, 10 runs)
  Run 1/10: 2 loops, think=0, tools={'tavily_search': 1} ← SKIPPED
  Run 2/10: 2 loops, think=0, tools={'tavily_search': 1} ← SKIPPED
  Run 3/10: 2 loops, think=0, tools={'tavily_search': 1} ← SKIPPED
  Run 4/10: 2 loops, think=0, tools={'tavily_search': 1} ← SKIPPED
  Run 5/10: 2 loops, think=0, tools={'tavily_search': 1} ← SKIPPED
  Run 6/10: 2 loops, think=0, tools={'tavily_search': 1} ← SKIPPED
  Run 7/10: 2 loops, think=0, tools={'tavily_search': 1} ← SKIPPED
  Run 8/10: 2 loops, think=0, tools={'tavily_search': 1} ← SKIPPED
  Run 9/10: 2 loops, think=0, tools={'tavily_search': 1} ← SKIPPED
  Run 10/10: 2 loops, think=0, tools={'tavily_search': 1} ← SKIPPED

Think Tool — Flash Lite
  Adherence:        0/10 runs used think (0%)
  Skipped:          10/10 runs (100%)
  Called first:     0/10 runs (0%)
  Avg times called: 0.0
  Avg loops:        2.0
  Avg tokens:       6551
  Sample sequences:
    Run 1: tavily_search
    Run 2: tavily_s

In [10]:
graph_think_pro = build_graph([tavily_search, think], prompts['think'], model=llm_pro)
print(f"Experiment 1b — Think Tool (Pro, {N_RUNS} runs)")
runs_think_pro = run_experiment(graph_think_pro, TASK, 'think')
summarize_adherence(runs_think_pro, 'think', 'Think Tool — Pro')

Experiment 1b — Think Tool (Pro, 10 runs)
  Run 1/10: 4 loops, think=3, tools={'think': 3, 'tavily_search': 3}
  Run 2/10: 5 loops, think=0, tools={'tavily_search': 4} ← SKIPPED
  Run 3/10: 2 loops, think=0, tools={'tavily_search': 1} ← SKIPPED
  Run 4/10: 3 loops, think=0, tools={'tavily_search': 2} ← SKIPPED
  Run 5/10: 3 loops, think=0, tools={'tavily_search': 2} ← SKIPPED
  Run 6/10: 4 loops, think=3, tools={'think': 3, 'tavily_search': 3}
  Run 7/10: 3 loops, think=1, tools={'tavily_search': 2, 'think': 1}
  Run 8/10: 4 loops, think=0, tools={'tavily_search': 3} ← SKIPPED
  Run 9/10: 5 loops, think=0, tools={'tavily_search': 4} ← SKIPPED
  Run 10/10: 4 loops, think=0, tools={'tavily_search': 3} ← SKIPPED

Think Tool — Pro
  Adherence:        3/10 runs used think (30%)
  Skipped:          7/10 runs (70%)
  Called first:     2/10 runs (20%)
  Avg times called: 0.7
  Avg loops:        3.7
  Avg tokens:       30249
  Sample sequences:
    Run 1: think → tavily_search → think → tavily_

---
### Experiment 2 — Classify (skippable but useful)

Returns a category the model could infer on its own. System prompt says: "You MUST use classify_query before doing anything else."

In [19]:
graph_classify = build_graph([tavily_search, classify_query], prompts['classify'])
print(f"Experiment 2 — Classify Tool (Flash Lite, {N_RUNS} runs)")
runs_classify_lite = run_experiment(graph_classify, TASK, 'classify_query')
summarize_adherence(runs_classify_lite, 'classify_query', 'Classify — Flash Lite')

Experiment 2 — Classify Tool (Flash Lite, 10 runs)
  Run 1/10: 3 loops, classify_query=1, tools={'classify_query': 1, 'tavily_search': 1}
  Run 2/10: 3 loops, classify_query=1, tools={'classify_query': 1, 'tavily_search': 1}
  Run 3/10: 3 loops, classify_query=1, tools={'classify_query': 1, 'tavily_search': 1}
  Run 4/10: 3 loops, classify_query=1, tools={'classify_query': 1, 'tavily_search': 1}
  Run 5/10: 3 loops, classify_query=1, tools={'classify_query': 1, 'tavily_search': 1}
  Run 6/10: 3 loops, classify_query=1, tools={'classify_query': 1, 'tavily_search': 1}
  Run 7/10: 3 loops, classify_query=1, tools={'classify_query': 1, 'tavily_search': 1}
  Run 8/10: 3 loops, classify_query=1, tools={'classify_query': 1, 'tavily_search': 1}
  Run 9/10: 3 loops, classify_query=1, tools={'classify_query': 1, 'tavily_search': 1}
  Run 10/10: 3 loops, classify_query=1, tools={'classify_query': 1, 'tavily_search': 1}

Classify — Flash Lite
  Adherence:        10/10 runs used classify_query (100

In [6]:
graph_classify_pro = build_graph([tavily_search, classify_query], prompts['classify'], model=llm_pro)
print(f"Experiment 2b — Classify Tool (Pro, {N_RUNS} runs)")
runs_classify_pro = run_experiment(graph_classify_pro, TASK, 'classify_query')
summarize_adherence(runs_classify_pro, 'classify_query', 'Classify — Pro')

Experiment 2b — Classify Tool (Pro, 10 runs)
  Run 1/10: 4 loops, classify_query=1, tools={'classify_query': 1, 'tavily_search': 2}
  Run 2/10: 4 loops, classify_query=1, tools={'classify_query': 1, 'tavily_search': 2}
  Run 3/10: 4 loops, classify_query=1, tools={'classify_query': 1, 'tavily_search': 2}
  Run 4/10: 3 loops, classify_query=1, tools={'classify_query': 1, 'tavily_search': 1}
  Run 5/10: 5 loops, classify_query=1, tools={'classify_query': 1, 'tavily_search': 3}
  Run 6/10: 4 loops, classify_query=1, tools={'classify_query': 1, 'tavily_search': 2}
  Run 7/10: 4 loops, classify_query=1, tools={'classify_query': 1, 'tavily_search': 2}
  Run 8/10: 3 loops, classify_query=1, tools={'classify_query': 1, 'tavily_search': 1}
  Run 9/10: 3 loops, classify_query=1, tools={'classify_query': 1, 'tavily_search': 1}
  Run 10/10: 4 loops, classify_query=1, tools={'classify_query': 1, 'tavily_search': 2}

Classify — Pro
  Adherence:        10/10 runs used classify_query (100%)
  Skipped:

---
### Experiment 3 — Plan (structurally useful)

Returns sub-questions that guide the searches. System prompt says: "You MUST use research_plan before performing any searches."

In [26]:
graph_plan = build_graph([tavily_search, research_plan], prompts['plan'])
print(f"Experiment 3 — Plan Tool (Flash Lite, {N_RUNS} runs)")
runs_plan_lite = run_experiment(graph_plan, TASK, 'research_plan')
summarize_adherence(runs_plan_lite, 'research_plan', 'Plan — Flash Lite')

Experiment 3 — Plan Tool (Flash Lite, 10 runs)
  Run 1/10: 3 loops, research_plan=1, tools={'research_plan': 1, 'tavily_search': 1}
  Run 2/10: 3 loops, research_plan=1, tools={'research_plan': 1, 'tavily_search': 1}
  Run 3/10: 3 loops, research_plan=1, tools={'research_plan': 1, 'tavily_search': 1}
  Run 4/10: 3 loops, research_plan=1, tools={'research_plan': 1, 'tavily_search': 1}
  Run 5/10: 3 loops, research_plan=1, tools={'research_plan': 1, 'tavily_search': 1}
  Run 6/10: 3 loops, research_plan=1, tools={'research_plan': 1, 'tavily_search': 1}
  Run 7/10: 3 loops, research_plan=1, tools={'research_plan': 1, 'tavily_search': 1}
  Run 8/10: 3 loops, research_plan=1, tools={'research_plan': 1, 'tavily_search': 1}
  Run 9/10: 3 loops, research_plan=1, tools={'research_plan': 1, 'tavily_search': 1}
  Run 10/10: 3 loops, research_plan=1, tools={'research_plan': 1, 'tavily_search': 1}

Plan — Flash Lite
  Adherence:        10/10 runs used research_plan (100%)
  Skipped:          0/10 r

In [7]:
graph_plan_pro = build_graph([tavily_search, research_plan], prompts['plan'], model=llm_pro)
print(f"Experiment 3b — Plan Tool (Pro, {N_RUNS} runs)")
runs_plan_pro = run_experiment(graph_plan_pro, TASK, 'research_plan')
summarize_adherence(runs_plan_pro, 'research_plan', 'Plan — Pro')

Experiment 3b — Plan Tool (Pro, 10 runs)
  Run 1/10: 5 loops, research_plan=1, tools={'research_plan': 1, 'tavily_search': 3}
  Run 2/10: 7 loops, research_plan=1, tools={'research_plan': 1, 'tavily_search': 5}
  Run 3/10: 4 loops, research_plan=1, tools={'research_plan': 1, 'tavily_search': 2}
  Run 4/10: 6 loops, research_plan=1, tools={'research_plan': 1, 'tavily_search': 4}
  Run 5/10: 5 loops, research_plan=1, tools={'research_plan': 1, 'tavily_search': 3}
  Run 6/10: 4 loops, research_plan=1, tools={'research_plan': 1, 'tavily_search': 2}
  Run 7/10: 4 loops, research_plan=1, tools={'research_plan': 1, 'tavily_search': 2}
  Run 8/10: 3 loops, research_plan=1, tools={'research_plan': 1, 'tavily_search': 1}
  Run 9/10: 6 loops, research_plan=1, tools={'research_plan': 1, 'tavily_search': 4}
  Run 10/10: 5 loops, research_plan=1, tools={'research_plan': 1, 'tavily_search': 3}

Plan — Pro
  Adherence:        10/10 runs used research_plan (100%)
  Skipped:          0/10 runs (0%)
  Ca

---
### Experiment 4 — Validate (critical for correctness)

Checks the answer for unsupported claims before responding. System prompt says: "You MUST use validate_answer before giving your final answer."

In [28]:
graph_validate = build_graph([tavily_search, validate_answer], prompts['validate'])
print(f"Experiment 4 — Validate Tool (Flash Lite, {N_RUNS} runs)")
runs_validate_lite = run_experiment(graph_validate, TASK, 'validate_answer')
summarize_adherence(runs_validate_lite, 'validate_answer', 'Validate — Flash Lite')

Experiment 4 — Validate Tool (Flash Lite, 10 runs)
  Run 1/10: 3 loops, validate_answer=1, tools={'tavily_search': 1, 'validate_answer': 1}
  Run 2/10: 3 loops, validate_answer=1, tools={'tavily_search': 1, 'validate_answer': 1}
  Run 3/10: 3 loops, validate_answer=1, tools={'tavily_search': 1, 'validate_answer': 1}
  Run 4/10: 3 loops, validate_answer=1, tools={'tavily_search': 1, 'validate_answer': 1}
  Run 5/10: 3 loops, validate_answer=1, tools={'tavily_search': 1, 'validate_answer': 1}
  Run 6/10: 3 loops, validate_answer=1, tools={'tavily_search': 1, 'validate_answer': 1}
  Run 7/10: 3 loops, validate_answer=1, tools={'tavily_search': 1, 'validate_answer': 1}
  Run 8/10: 3 loops, validate_answer=1, tools={'tavily_search': 1, 'validate_answer': 1}
  Run 9/10: 3 loops, validate_answer=1, tools={'tavily_search': 1, 'validate_answer': 1}
  Run 10/10: 3 loops, validate_answer=1, tools={'tavily_search': 1, 'validate_answer': 1}

Validate — Flash Lite
  Adherence:        10/10 runs used

In [8]:
graph_validate_pro = build_graph([tavily_search, validate_answer], prompts['validate'], model=llm_pro)
print(f"Experiment 4b — Validate Tool (Pro, {N_RUNS} runs)")
runs_validate_pro = run_experiment(graph_validate_pro, TASK, 'validate_answer')
summarize_adherence(runs_validate_pro, 'validate_answer', 'Validate — Pro')

Experiment 4b — Validate Tool (Pro, 10 runs)
  Run 1/10: 5 loops, validate_answer=1, tools={'tavily_search': 3, 'validate_answer': 1}
  Run 2/10: 3 loops, validate_answer=0, tools={'tavily_search': 2} ← SKIPPED
  Run 3/10: 2 loops, validate_answer=0, tools={'tavily_search': 1} ← SKIPPED
  Run 4/10: 3 loops, validate_answer=0, tools={'tavily_search': 2} ← SKIPPED
  Run 5/10: 3 loops, validate_answer=0, tools={'tavily_search': 2} ← SKIPPED
  Run 6/10: 4 loops, validate_answer=0, tools={'tavily_search': 3} ← SKIPPED
  Run 7/10: 3 loops, validate_answer=0, tools={'tavily_search': 2} ← SKIPPED
  Run 8/10: 4 loops, validate_answer=0, tools={'tavily_search': 3} ← SKIPPED
  Run 9/10: 4 loops, validate_answer=0, tools={'tavily_search': 3} ← SKIPPED
  Run 10/10: 4 loops, validate_answer=0, tools={'tavily_search': 3} ← SKIPPED

Validate — Pro
  Adherence:        1/10 runs used validate_answer (10%)
  Skipped:          9/10 runs (90%)
  Called first:     0/10 runs (0%)
  Avg times called: 0.1
  Av

---
### Experiment 5 — Log (policy-mandated)

Audit logging. The model doesn't need the return value to finish the task. System prompt says: "You MUST call log_query before performing any search. This is a mandatory audit requirement."

In [21]:
graph_log = build_graph([tavily_search, log_query], prompts['log'])
print(f"Experiment 5 — Log Tool (Flash Lite, {N_RUNS} runs)")
runs_log_lite = run_experiment(graph_log, TASK, 'log_query')
summarize_adherence(runs_log_lite, 'log_query', 'Log — Flash Lite')

Experiment 5 — Log Tool (Flash Lite, 10 runs)
  Run 1/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}
  Run 2/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}
  Run 3/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}
  Run 4/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}
  Run 5/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}
  Run 6/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}
  Run 7/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}
  Run 8/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}
  Run 9/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}
  Run 10/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}

Log — Flash Lite
  Adherence:        10/10 runs used log_query (100%)
  Skipped:          0/10 runs (0%)
  Called first:     10/10 runs (100%)
  Avg times called: 1.0
  Avg loops:   

In [9]:
graph_log_pro = build_graph([tavily_search, log_query], prompts['log'], model=llm_pro)
print(f"Experiment 5b — Log Tool (Pro, {N_RUNS} runs)")
runs_log_pro = run_experiment(graph_log_pro, TASK, 'log_query')
summarize_adherence(runs_log_pro, 'log_query', 'Log — Pro')

Experiment 5b — Log Tool (Pro, 10 runs)
  Run 1/10: 4 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 2}
  Run 2/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}
  Run 3/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}
  Run 4/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}
  Run 5/10: 4 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 2}
  Run 6/10: 4 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 2}
  Run 7/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}
  Run 8/10: 4 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 2}
  Run 9/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}
  Run 10/10: 4 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 2}

Log — Pro
  Adherence:        10/10 runs used log_query (100%)
  Skipped:          0/10 runs (0%)
  Called first:     10/10 runs (100%)
  Avg times called: 1.0
  Avg loops:        3.5
  Av

---
### Experiment 6 — Task phrasing

Does the skip rate change with how the task is worded? Testing the most-skipped tool (think) with three phrasings: original, simple, and complex.

In [ ]:
TASK_VARIANTS = {
    'original': "What are the key differences between LangGraph and CrewAI for building multi-agent systems in 2026?",
    'simple':   "LangGraph vs CrewAI — which should I use?",
    'complex':  "I'm building a multi-agent customer support system in Python. Compare LangGraph and CrewAI on architecture, production readiness, observability, multi-model support, and community size as of 2026. Which would you recommend and why?",
}

# Use think tool (likely most-skipped) with Flash Lite
task_variant_results = {}
for variant_name, variant_task in TASK_VARIANTS.items():
    graph = build_graph([tavily_search, think], prompts['think'])
    print(f"\nTask variant: {variant_name}")
    runs = run_experiment(graph, variant_task, 'think')
    summarize_adherence(runs, 'think', f'Think — {variant_name}')
    task_variant_results[variant_name] = runs


Task variant: original
  Run 1/10: 2 loops, think=0, tools={'tavily_search': 1} ← SKIPPED
  Run 2/10: 2 loops, think=0, tools={'tavily_search': 1} ← SKIPPED
  Run 3/10: 2 loops, think=0, tools={'tavily_search': 1} ← SKIPPED
  Run 4/10: 2 loops, think=0, tools={'tavily_search': 1} ← SKIPPED
  Run 5/10: 2 loops, think=0, tools={'tavily_search': 1} ← SKIPPED
  Run 6/10: 2 loops, think=0, tools={'tavily_search': 1} ← SKIPPED
  Run 7/10: 2 loops, think=0, tools={'tavily_search': 1} ← SKIPPED
  Run 8/10: 2 loops, think=0, tools={'tavily_search': 1} ← SKIPPED
  Run 9/10: 2 loops, think=0, tools={'tavily_search': 1} ← SKIPPED
  Run 10/10: 2 loops, think=0, tools={'tavily_search': 1} ← SKIPPED

Think — original
  Adherence:        0/10 runs used think (0%)
  Skipped:          10/10 runs (100%)
  Called first:     0/10 runs (0%)
  Avg times called: 0.0
  Avg loops:        2.0
  Avg tokens:       6535
  Sample sequences:
    Run 1: tavily_search
    Run 2: tavily_search
    Run 3: tavily_search


---
### Experiment 7 — Few-shot examples

Does showing the model a workflow example of using the tool increase adherence? Testing on think (worst skip rate) and log (baseline comparison).

In [14]:
# Few-shot prompt for think tool
prompt_fewshot_think = SystemMessage(content=(
    'You are a research assistant. Search for accurate, up-to-date information. '
    'Before EVERY search, you MUST use the think tool to reason about '
    'what you know so far and what you still need to find.\n\n'
    'Example workflow:\n'
    '1. User asks: "What is the best database for AI agents?"\n'
    '2. You call think("I need to understand what databases are commonly used with AI agents. '
    'I should search for recent comparisons and benchmarks.")\n'
    '3. You call tavily_search("best database AI agents 2026 comparison")\n'
    '4. You call think("The results mention pgvector and ChromaDB. I should search for '
    'production experiences with each to compare.")\n'
    '5. You call tavily_search("pgvector vs chromadb production AI agents")\n'
    '6. You call think("I now have enough information. pgvector is better for teams already '
    'on Postgres, ChromaDB is simpler for prototyping.")\n'
    '7. You give your final answer.\n\n'
    'Always think before searching. Always think after searching. Cite your sources with URLs.'
))

# Few-shot prompt for log tool
prompt_fewshot_log = SystemMessage(content=(
    'You are a research assistant. Search for accurate, up-to-date information. '
    'For compliance, you MUST call log_query before performing any search.\n\n'
    'Example workflow:\n'
    '1. User asks: "What is the best database for AI agents?"\n'
    '2. You call log_query(query="What is the best database for AI agents?", timestamp="2026-03-30T12:00:00")\n'
    '3. You receive: "LOGGED — ID: a1b2c3d4e5f6. Proceed with research."\n'
    '4. NOW you call tavily_search("best database AI agents 2026")\n'
    '5. You give your final answer.\n\n'
    'The log_query call must happen FIRST, before any search. This is a mandatory audit requirement. '
    'Cite your sources with URLs.'
))

print("Few-shot prompts ready.")

Few-shot prompts ready.


In [24]:
# Few-shot: Think tool
graph_fs_think = build_graph([tavily_search, think], prompt_fewshot_think)
print(f"Experiment 7a — Think + Few-Shot (Flash Lite, {N_RUNS} runs)")
runs_fs_think = run_experiment(graph_fs_think, TASK, 'think')
summarize_adherence(runs_fs_think, 'think', 'Think + Few-Shot — Flash Lite')

Experiment 7a — Think + Few-Shot (Flash Lite, 10 runs)
  Run 1/10: 3 loops, think=1, tools={'think': 1, 'tavily_search': 1}
  Run 2/10: 3 loops, think=1, tools={'think': 1, 'tavily_search': 1}
  Run 3/10: 4 loops, think=2, tools={'think': 2, 'tavily_search': 1}
  Run 4/10: 3 loops, think=1, tools={'think': 1, 'tavily_search': 1}
  Run 5/10: 4 loops, think=2, tools={'think': 2, 'tavily_search': 1}
  Run 6/10: 3 loops, think=1, tools={'think': 1, 'tavily_search': 1}
  Run 7/10: 3 loops, think=1, tools={'think': 1, 'tavily_search': 1}
  Run 8/10: 3 loops, think=1, tools={'think': 1, 'tavily_search': 1}
  Run 9/10: 3 loops, think=1, tools={'think': 1, 'tavily_search': 1}
  Run 10/10: 3 loops, think=1, tools={'think': 1, 'tavily_search': 1}

Think + Few-Shot — Flash Lite
  Adherence:        10/10 runs used think (100%)
  Skipped:          0/10 runs (0%)
  Called first:     10/10 runs (100%)
  Avg times called: 1.2
  Avg loops:        3.2
  Avg tokens:       9888
  Sample sequences:
    Run 

In [15]:
# Few-shot: Log tool
graph_fs_log = build_graph([tavily_search, log_query], prompt_fewshot_log)
print(f"Experiment 7b — Log + Few-Shot (Flash Lite, {N_RUNS} runs)")
runs_fs_log = run_experiment(graph_fs_log, TASK, 'log_query')
summarize_adherence(runs_fs_log, 'log_query', 'Log + Few-Shot — Flash Lite')

Experiment 7b — Log + Few-Shot (Flash Lite, 10 runs)
  Run 1/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}
  Run 2/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}
  Run 3/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}
  Run 4/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}
  Run 5/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}
  Run 6/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}
  Run 7/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}
  Run 8/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}
  Run 9/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}
  Run 10/10: 3 loops, log_query=1, tools={'log_query': 1, 'tavily_search': 1}

Log + Few-Shot — Flash Lite
  Adherence:        10/10 runs used log_query (100%)
  Skipped:          0/10 runs (0%)
  Called first:     10/10 runs (100%)
  Avg times called: 1

In [16]:
# Few-shot with Pro model for comparison
graph_fs_think_pro = build_graph([tavily_search, think], prompt_fewshot_think, model=llm_pro)
print(f"Experiment 7c — Think + Few-Shot (Pro, {N_RUNS} runs)")
runs_fs_think_pro = run_experiment(graph_fs_think_pro, TASK, 'think')
summarize_adherence(runs_fs_think_pro, 'think', 'Think + Few-Shot — Pro')

Experiment 7c — Think + Few-Shot (Pro, 10 runs)
  Run 1/10: 2 loops, think=1, tools={'think': 1, 'tavily_search': 1}
  Run 2/10: 4 loops, think=0, tools={'tavily_search': 3} ← SKIPPED
  Run 3/10: 4 loops, think=3, tools={'think': 3, 'tavily_search': 3}
  Run 4/10: 3 loops, think=2, tools={'think': 2, 'tavily_search': 2}
  Run 5/10: 4 loops, think=3, tools={'think': 3, 'tavily_search': 3}
  Run 6/10: 4 loops, think=3, tools={'think': 3, 'tavily_search': 3}
  Run 7/10: 5 loops, think=3, tools={'think': 3, 'tavily_search': 3}
  Run 8/10: 4 loops, think=3, tools={'think': 3, 'tavily_search': 3}
  Run 9/10: 3 loops, think=2, tools={'think': 2, 'tavily_search': 2}
  Run 10/10: 2 loops, think=1, tools={'think': 1, 'tavily_search': 1}

Think + Few-Shot — Pro
  Adherence:        9/10 runs used think (90%)
  Skipped:          1/10 runs (10%)
  Called first:     9/10 runs (90%)
  Avg times called: 2.1
  Avg loops:        3.5
  Avg tokens:       24074
  Sample sequences:
    Run 1: think → tavily_

---
## Results

All experiments side by side.

In [29]:
all_experiments = {
    # (label, runs, tool_name, model, prompt_type)
    'Think (Lite)':           (runs_think_lite, 'think', 'Flash Lite', 'basic'),
    'Think (Pro)':            (runs_think_pro, 'think', 'Pro', 'basic'),
    'Think+FewShot (Lite)':   (runs_fs_think, 'think', 'Flash Lite', 'few-shot'),
    'Think+FewShot (Pro)':    (runs_fs_think_pro, 'think', 'Pro', 'few-shot'),
    'Classify (Lite)':        (runs_classify_lite, 'classify_query', 'Flash Lite', 'basic'),
    'Classify (Pro)':         (runs_classify_pro, 'classify_query', 'Pro', 'basic'),
    'Plan (Lite)':            (runs_plan_lite, 'research_plan', 'Flash Lite', 'basic'),
    'Plan (Pro)':             (runs_plan_pro, 'research_plan', 'Pro', 'basic'),
    'Validate (Lite)':        (runs_validate_lite, 'validate_answer', 'Flash Lite', 'basic'),
    'Validate (Pro)':         (runs_validate_pro, 'validate_answer', 'Pro', 'basic'),
    'Log (Lite)':             (runs_log_lite, 'log_query', 'Flash Lite', 'basic'),
    'Log (Pro)':              (runs_log_pro, 'log_query', 'Pro', 'basic'),
    'Log+FewShot (Lite)':     (runs_fs_log, 'log_query', 'Flash Lite', 'few-shot'),
}

print(f"{'Experiment':<25} {'Model':<12} {'Prompt':<10} {'Adherence':<12} {'Skip%':<8} {'Avg Tokens':<12} {'Avg Loops'}")
print('-' * 95)

for label, (runs, tool_name, model, prompt_type) in all_experiments.items():
    ok = [r for r in runs if not r.get('crashed')]
    if not ok:
        print(f"{label:<25} {model:<12} {prompt_type:<10} {'CRASHED':<12}")
        continue

    used = sum(1 for r in ok if not r['mandatory_skipped'])
    skip_pct = sum(1 for r in ok if r['mandatory_skipped']) / len(ok) * 100
    avg_tokens = statistics.mean([r['total_tokens'] for r in ok])
    avg_loops = statistics.mean([r['loops'] for r in ok])

    print(f"{label:<25} {model:<12} {prompt_type:<10} {used}/{len(ok):<10} {skip_pct:<8.0f} {avg_tokens:<12.0f} {avg_loops:.1f}")

Experiment                Model        Prompt     Adherence    Skip%    Avg Tokens   Avg Loops
-----------------------------------------------------------------------------------------------
Think (Lite)              Flash Lite   basic      0/10         100      6551         2.0
Think (Pro)               Pro          basic      3/10         70       30249        3.7
Think+FewShot (Lite)      Flash Lite   few-shot   10/10         0        9888         3.2
Think+FewShot (Pro)       Pro          few-shot   9/10         10       24074        3.5
Classify (Lite)           Flash Lite   basic      10/10         0        8304         3.0
Classify (Pro)            Pro          basic      10/10         0        16226        3.8
Plan (Lite)               Flash Lite   basic      10/10         0        8872         3.0
Plan (Pro)                Pro          basic      10/10         0        40418        4.9
Validate (Lite)           Flash Lite   basic      10/10         0        12555        3.0
Va

---
## What I found

- Classify, plan, and log: 100% adherence on both models. The model can't skip them and still finish the task.
- Think: 0% on Flash Lite, 30% on Flash. Returns nothing useful. The model skips it because skipping doesn't break anything.
- Validate: 100% on Flash Lite, 10% on Flash. The larger model is more confident and skips the safety check.
- Few-shot examples fixed think on Flash Lite (0% to 100%) but Flash still skipped 1 in 10 (30% to 90%).
- Task phrasing made no difference. Think got skipped regardless of how the question was worded.
- The pattern: the model evaluates whether it needs the tool's output to continue. If it doesn't, the "you MUST" instruction gets dropped. Stronger models skip more, not less.